In [1]:
import logging
import joblib
import mlflow
from pathlib import Path
import pandas as pd

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

In [2]:
MODEL_NAME = "heart_disease_pred_model"
MODEL_STAGE = "None"  # since you're not using staging yet

LOCAL_MODEL_PATH = Path("../models/heart_disease_pred_model.pkl")

In [3]:
def load_model_from_mlflow():
    try:
        logger.info("Trying MLflow Registry...")

        tracking_path = Path("../mlruns").resolve()
        mlflow.set_tracking_uri(f"file:///{tracking_path.as_posix()}")

        print("MLflow URI:", mlflow.get_tracking_uri())

        model_uri = f"models:/{MODEL_NAME}/{MODEL_STAGE}"
        model = mlflow.sklearn.load_model(model_uri)

        logger.info("Loaded from MLflow")
        return model

    except Exception as e:
        logger.warning(f"MLflow load failed: {e}")
        return None

In [4]:
def load_model_from_local():
    try:
        logger.info("Trying local model...")

        if not LOCAL_MODEL_PATH.exists():
            raise FileNotFoundError(f"{LOCAL_MODEL_PATH} not found")

        model = joblib.load(LOCAL_MODEL_PATH)

        logger.info("Loaded from local file")
        return model

    except Exception as e:
        logger.error(f"Local load failed: {e}")
        raise

In [5]:
def load_model():
    model = load_model_from_mlflow()

    if model is not None:
        return model

    logger.info("Falling back to local...")
    return load_model_from_local()

In [6]:
model = load_model()

print("Model type:", type(model))

2026-05-02 19:15:47,228 - INFO - Trying MLflow Registry...
c:\Bhooshaan_Local\BITS Local\Sem2\Assignments\MLOps\MLOps Assignment\MLOps-Assignment\mlops_env\Lib\site-packages\mlflow\store\artifact\utils\models.py:32: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/2.12.2/model-registry.html#migrating-from-stages
  latest = client.get_latest_versions(name, None if stage is None else [stage])


MLflow URI: file:///C:/Bhooshaan_Local/BITS Local/Sem2/Assignments/MLOps/MLOps Assignment/MLOps-Assignment/mlruns


2026-05-02 19:15:48,983 - INFO - Loaded from MLflow


Model type: <class 'sklearn.pipeline.Pipeline'>


In [7]:
def predict_heart_disease(data: dict) -> dict:
    try:
        # Convert to DataFrame
        df = pd.DataFrame([data])

        # Prediction
        pred = model.predict(df)[0]

        # Confidence
        if hasattr(model, "predict_proba"):
            probs = model.predict_proba(df)[0]
            confidence = probs[pred]
        else:
            confidence = None

        result = {
            "prediction": int(pred),
            "confidence": float(round(confidence, 4)) if confidence is not None else None
        }

        logger.info(f"Prediction result: {result}")

        return result

    except Exception as e:
        logger.error(f"Prediction failed: {e}")
        raise

In [8]:
healthy_sample = {
    "age": 40,
    "sex": 0,
    "cp": 1,           # less severe chest pain
    "trestbps": 110,
    "chol": 180,
    "fbs": 0,
    "restecg": 0,
    "thalach": 170,    # high max heart rate (good sign)
    "exang": 0,
    "oldpeak": 0.0,    # no ST depression
    "slope": 2,
    "ca": 0,
    "thal": 2
}

disease_sample = {
    "age": 65,
    "sex": 1,
    "cp": 4,           # severe chest pain
    "trestbps": 160,
    "chol": 300,
    "fbs": 1,
    "restecg": 2,
    "thalach": 100,    # low max heart rate
    "exang": 1,
    "oldpeak": 4.0,    # strong ST depression
    "slope": 0,
    "ca": 3,
    "thal": 3
}


print("Healthy Sample Prediction:")
prediction = predict_heart_disease(healthy_sample).get("prediction")
confidence = predict_heart_disease(healthy_sample).get("confidence")
print({
    "prediction": int(prediction),
    "confidence": float(round(confidence, 4))
})


print("Disease Sample Prediction:")
prediction = predict_heart_disease(disease_sample).get("prediction")
confidence = predict_heart_disease(disease_sample).get("confidence")
print({
    "prediction": int(prediction),
    "confidence": float(round(confidence, 4))
})


2026-05-02 19:15:53,188 - INFO - Prediction result: {'prediction': 0, 'confidence': 0.8252}
2026-05-02 19:15:53,217 - INFO - Prediction result: {'prediction': 0, 'confidence': 0.8252}
2026-05-02 19:15:53,257 - INFO - Prediction result: {'prediction': 1, 'confidence': 0.8709}
2026-05-02 19:15:53,288 - INFO - Prediction result: {'prediction': 1, 'confidence': 0.8709}


Healthy Sample Prediction:
{'prediction': 0, 'confidence': 0.8252}
Disease Sample Prediction:
{'prediction': 1, 'confidence': 0.8709}
